# Q1 유지 · 자동 확대 / 프롬프트 분리 비교 — dev240

**L4/A100 새 런타임 → 기존 Secret GITHUB_TOKEN 허용 → 모두 실행.** Drive 인증 외 박스 그리기·정답 입력은 없습니다. 기존 dev 기준선·정상 참고 사진·원본 ZIP을 자동으로 읽고 새 전용 폴더에 저장합니다.

| 조건 | 대상 사진 | Q2/Q3 문구 |
|---|---|---|
| control | 기존 타일 예산3 | 현재 Q2 정의 / 기존 Q3 |
| zoom_only | 원본에서 자동 타일 예산6 | control과 동일 |
| prompt_only | 기존 타일 예산3 | 짧은 결함 증거 설명 |
| zoom_prompt | 원본에서 자동 타일 예산6 | prompt_only와 동일 |

기존 직접 Q2/Q3 기준선도 `cached_baseline`으로 함께 표시합니다(Q3 직접 답만 새로 생성). **Q1은 모든 조건에서 기존 답 그대로이며 신규 추론0회**입니다. 정상 참고 사진은 같은 파일·타일 예산3으로 고정합니다. 타일은 원본에서 자동 분할되며 큰 정사각형은 확대 조건에서2×2+전체 thumbnail로 바뀔 수 있습니다. 작은 사진은 타일 수가 그대로일 수 있고 실제 수를 기록합니다. 사람이 ROI를 지정하지 않습니다.

모든 조건에서 Q3는 유무 질문→있다고 답했을 때만 위치 질문입니다. 확대 조건에는 확대를 유도하는 별도 문장을 추가하지 않아 입력 효과를 분리합니다. 프롬프트 조건만 문구를 바꾸며 같은 선택지·같은 경로를 유지합니다. 서로 다른 문항의 답은 공유하지 않습니다.

신규 Q2/Q3 호출은 **2,160~3,120회/240장**이고 조건별 실제 비용·시간을 기록합니다. 확대는 토큰/메모리를 더 씁니다. 실제 총시간·성능은 아직 미측정입니다. 20호출마다 저장하며 같은 추론 셀을 재실행하면 성공 호출을 재사용하고 실패/무효 답은 재시도합니다. 메모리 부족 때 예산을 몰래 낮추지 않고 실패 조건을 제외합니다. 코드/입력 변경 시 새 출력 폴더가 필요합니다.

**이미 여러 번 분석한 dev의 탐색 실험**입니다. gold를 재튜닝하거나 설정을 자동 선택하지 않습니다. Q3는 기존 이미지 정답으로 결함 유무만 자동 채점하며 실제9구역 위치 정확도는 `null`입니다. 정상 오탐과 결함 놓침을 함께 확인하세요. 과거 결과와 원본은 보존합니다.

출력: `MyDrive/lvlm_insplad_results/gold_v2_q23_zoom_prompt_dev_v1/`.
마지막 셀에서 `dev_q23_zoom_prompt_results.zip`을 다운로드합니다.


In [ ]:
REPO = "arnold6444/lvlm_insplad"
EXECUTION_COMMIT = "7faafeb63f80ada1b2d583929a309bb1931884d2"
DEV_FOLDER = "lvlm_insplad_qa/gold_v2_dev"
DEV_BASELINES_FOLDER = "lvlm_insplad_results/gold_v2_dev_baselines"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_q23_zoom_prompt_dev_v1"
N_IMAGES = 240
SEED = 42
CHECKPOINT_EVERY = 20
UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""


In [ ]:
# Drive 연결 + repo clone + 패키지
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
DEV_DIR = DRIVE_ROOT / DEV_FOLDER
STAGE1_DIR = DRIVE_ROOT / DEV_BASELINES_FOLDER / "predictions"
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
if not RUN_ROOT.name.startswith("gold_v2_q23_zoom_prompt_dev"):
    raise ValueError("새 실험 전용 RUN_FOLDER를 사용하세요.")
REPO_DIR = Path("/content/lvlm_q23_zoom_prompt_code")

def run(cmd, *, env=None, check=True):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
# Fetch exactly the locked object, never a moving branch.
if not REPO_DIR.exists():
    REPO_DIR.mkdir(parents=True)
    run(["git", "init", REPO_DIR])
    run(["git", "-C", REPO_DIR, "remote", "add", "origin", f"https://github.com/{REPO}.git"])
    run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", EXECUTION_COMMIT], env=git_env)
    run(["git", "-C", REPO_DIR, "checkout", "--detach", EXECUTION_COMMIT])
else:
    existing_remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
    if existing_remote != f"https://github.com/{REPO}.git":
        raise RuntimeError("실행 폴더의 저장소가 다릅니다. 새 런타임을 사용하세요.")
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if actual_revision != EXECUTION_COMMIT:
    raise RuntimeError("고정 커밋과 다른 코드입니다. 새 런타임을 사용하세요.")
if subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO_DIR, text=True).strip():
    raise RuntimeError("고정 코드가 수정됐습니다. 새 런타임을 사용하세요.")
del auth, git_env
run([sys.executable,"-m","pip","install","-q","transformers==5.14.0","accelerate==1.14.0",
     "sentencepiece==0.2.2","pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
from verify_gold_revision import verify as verify_gold_revision
verify_gold_revision(REPO_DIR, EXECUTION_COMMIT)
import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")
if torch.cuda.get_device_properties(0).total_memory < 20 * 1024**3:
    raise RuntimeError("VRAM 20GB 이상 L4/A100이 필요합니다.")


In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음)", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("SUP:", SUP_ZIP); print("UNSUP:", UNSUP_ZIP)



import run_gold_v2_q23_zoom_prompt_dev as experiment
summary = json.loads((DEV_DIR / "build_summary.json").read_text(encoding="utf-8"))
if summary.get("quality_status") != "FROZEN":
    raise RuntimeError("FROZEN dev 세트가 필요합니다.")
DEV_QA = DEV_DIR / summary["file"]
REFS_CSV = DEV_DIR / "refs/retrieved_refs.csv"
need = [DEV_QA, REFS_CSV, STAGE1_DIR / experiment.MODEL / "normal_ref_retrieved_diffdate.jsonl"]
missing = [str(p) for p in need if not p.is_file()]
if missing:
    raise FileNotFoundError("기존 dev 입력을 찾지 못했습니다: " + ", ".join(missing))
rows, _ = experiment.refined.select_rows(experiment.agent.read_jsonl(DEV_QA), N_IMAGES, SEED)
experiment.common.read_references(REFS_CSV, rows)
experiment.baseline(need[-1], rows)
print(f"입력 확인: {len(rows)//2}장, Q1 추가 호출0, 신규 Q2/Q3 {N_IMAGES*9}~{N_IMAGES*13}회")
print("정상 참고 예산3 고정 / 대상 예산3 대6. 새 출력:", RUN_ROOT)


In [ ]:
# 추론+자동 집계. 중단되면 이 셀을 다시 실행하세요.
RUNNER_PATH = REPO_DIR / "scripts/run_gold_v2_q23_zoom_prompt_dev.py"
run([sys.executable, RUNNER_PATH, "--dev-qa", DEV_QA, "--stage1-dir", STAGE1_DIR,
     "--refs-csv", REFS_CSV, "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP,
     "--out-dir", RUN_ROOT, "--n-images", str(N_IMAGES), "--seed", str(SEED),
     "--checkpoint-every", str(CHECKPOINT_EVERY)])


In [ ]:
# 조건별 효과를 따로 확인. 자동 설정 선택 없음.
status = json.loads((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))
meta = json.loads((RUN_ROOT / "run_metadata.json").read_text(encoding="utf-8"))
display(pd.DataFrame(status["policies"]))
print("신규:", meta.get("called_this_run"), "재사용:", meta.get("reused_calls"),
      "오류:", meta.get("row_errors"), "무효:", meta.get("invalid_answers"))
if not status["complete"] or not meta["complete"]:
    raise RuntimeError("미완료 조건이 있습니다. 실패/무효 조건은 비교에서 제외됩니다. 추론 셀을 재실행하세요.")
answers = pd.read_csv(RUN_ROOT / "answers.csv")
q1 = answers[answers.question_type == "anomaly_detection"]
assert len(q1) == meta["images"] * len(meta["policies"])
assert (q1.predicted_choice == q1.cached_baseline_choice).all() and meta["q1_inference_calls"] == 0
print("Q1 유지:", len(q1), "답 동일 / 신규0회")
lb = pd.read_csv(RUN_ROOT / "leaderboard.csv")
display(lb[["policy", "q1_anomaly_f1", "q1_balanced_accuracy", "q2_defect_macro_f1", "q2_defect_balanced_accuracy",
    "joint_accuracy", "false_defect_rate", "q3_presence_balanced_accuracy", "q3_normal_false_region_rate",
    "q3_anomaly_false_none_rate", "incremental_tool_calls", "tool_latency_s"]].round(4))
print("Q2 결함별 정답 / Q3 결함별 놓침·정상 오탐 (특히 missing-cap)")
display(pd.read_csv(RUN_ROOT / "q2_by_defect.csv").round(4))
display(pd.read_csv(RUN_ROOT / "q3_by_defect.csv"))
print("짝지은 bootstrap CI: control→zoom_only는 입력 효과, control→prompt_only는 문구 효과")
sig = pd.read_csv(RUN_ROOT / "significance.csv")
display(sig[sig.metric.isin(["q2_defect_macro_f1", "joint_accuracy", "q3_presence_balanced_accuracy",
                            "q3_normal_false_region_rate", "q3_anomaly_false_none_rate"])].round(4))
print("원본 해상도와 실제 타일 수 (thumbnail 포함)")
manifest = pd.read_csv(RUN_ROOT / "image_manifest.csv")
display(manifest.groupby(["role", "standard_tiles", "zoom_tiles"]).size().rename("images").reset_index())
print("자산별 결과")
display(pd.read_csv(RUN_ROOT / "metrics_by_asset.csv").round(4))
print("항상없음/항상영역 대조")
display(pd.read_csv(RUN_ROOT / "q3_constant_baselines.csv").round(4))
print("기존 dev 재검증: 탐색 결과 / 실제 위치 정확도:", meta["q3_location_accuracy"])
print("결과 경로:", RUN_ROOT)


In [ ]:
# Drive 결과는 보존하고 결과 ZIP만 다운로드
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = Path("/content/dev_q23_zoom_prompt_results.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for path in sorted(RUN_ROOT.rglob("*")):
        relative = path.relative_to(RUN_ROOT)
        if path.is_file() and "_images" not in relative.parts:
            z.write(path, relative)
files.download(str(archive))
